# Azure OpenAI 연결 테스트

FastAPI 앱과 별개로, Azure OpenAI(GPT-5) 자격증명/배포가 제대로 동작하는지 단일 노트북에서 빠르게 확인하기 위한 템플릿입니다.

## 사전 준비
1. 프로젝트 루트(`azure-doc-ai-service/`)에 `.env` 파일이 있어야 합니다. 없다면 `.env.example`을 복사해서 실제 값을 채우세요.
2. 커널 실행 전, 아래 패키지가 설치되어 있어야 합니다.

```bash
.\.venv\Scripts\python.exe -m pip install jupyter ipykernel
```

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/)에 있는 .env를 로드
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"]
AZURE_OPENAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")

print("ENDPOINT:", AZURE_OPENAI_ENDPOINT)
print("DEPLOYMENT:", AZURE_OPENAI_DEPLOYMENT_NAME)
print("API VERSION:", AZURE_OPENAI_API_VERSION)

In [ ]:
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

## 단발성 호출 테스트

In [ ]:
response = client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 어시스턴트입니다."},
        {"role": "user", "content": "1부터 5까지 더한 값은?"},
    ],
    max_completion_tokens=200,
)

print(response.choices[0].message.content)
print("---")
print("finish_reason:", response.choices[0].finish_reason)
print("usage:", response.usage)

## 반복 테스트용 헬퍼 함수

프롬프트만 바꿔가며 빠르게 호출해볼 때 사용하세요.

In [ ]:
def ask(prompt: str, system: str = "당신은 간결하게 답하는 어시스턴트입니다.", max_output_tokens: int = 500) -> str:
    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        max_completion_tokens=max_output_tokens,
    )
    return response.choices[0].message.content or ""

In [ ]:
print(ask("파이썬에서 리스트와 튜플의 차이를 한 문장으로 설명해줘."))

## 스트리밍 테스트 (선택)

실시간으로 토큰이 오는지 확인하고 싶을 때 사용하세요.

In [ ]:
stream = client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT_NAME,
    messages=[{"role": "user", "content": "바다에 대한 짧은 시를 써줘."}],
    max_completion_tokens=300,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        print(delta, end="", flush=True)